# Demo: Using kb-mcp with hep-multiagent

In [ ]:
# !pip install -q --force-reinstall git+https://github.com/HEP-KE/mcp-ke.git
!pip install -q --force-reinstall git+https://github.com/HEP-KE/HEP-multiagent.git
#install last since it needs mcp<1.23.0 but 1.26.0 got installed
!pip install -q --force-reinstall git+https://github.com/HEP-KE/kb-mcp.git

### Set up for KB MCP

Set up paths and choose which papers to download.

In [2]:
import os
import subprocess
import sys
from pathlib import Path

# Paths (stores data in current directory)
DATA_DIR = Path.cwd() / "data"
DB_PATH = DATA_DIR / "kb.db"
PAPERS_DIR = DATA_DIR / "papers"

# Papers to download (arXiv ID, title)
PAPERS = [
    ("1807.06209", "Planck 2018 cosmological parameters"),
    ("2007.08991", "eBOSS cosmological results"),
    ("1502.01589", "Planck 2015 cosmological results"),
]

print(f"Database: {DB_PATH}")
print(f"Papers: {PAPERS_DIR}")

Database: /Users/celsloaner/Desktop/hep-multiagent-v2/tests/data/kb.db
Papers: /Users/celsloaner/Desktop/hep-multiagent-v2/tests/data/papers


Initialize an empty SQLite database with the kb-mcp schema.

In [3]:
from kb_mcp.kb.db_models import Base
from sqlalchemy import create_engine

# Create directories
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
PAPERS_DIR.mkdir(parents=True, exist_ok=True)

# Create database
engine = create_engine(f"sqlite:///{DB_PATH}")
Base.metadata.create_all(engine)

print(f"Created database: {DB_PATH}")

Created database: /Users/celsloaner/Desktop/hep-multiagent-v2/tests/data/kb.db


Fetch PDFs from arXiv and extract text.

In [4]:
from hep_multiagent.features.arxiv_fetch import download_full_text

for arxiv_id, title in PAPERS:
    txt_path = PAPERS_DIR / f"{arxiv_id}.txt"
    if txt_path.exists():
        print(f"{arxiv_id} - already downloaded")
    else:
        print(f"[downloading] {arxiv_id}: {title}")
        download_full_text(arxiv_id, str(PAPERS_DIR))

print(f"\nDownloaded {len(list(PAPERS_DIR.glob('*.txt')))} papers")

1807.06209 - already downloaded
2007.08991 - already downloaded
1502.01589 - already downloaded

Downloaded 3 papers


Ingest the downloaded papers into kb-mcp.

In [5]:
# NOTE: Embeddings required for kb_search to work with SQLite.
# Using --no-embed causes kb_search to crash (KeyError: 'total_results')
# because SQLite doesn't support full-text search.

os.environ["SQLITE_DB_PATH"] = str(DB_PATH)

for arxiv_id, _ in PAPERS:
    txt_path = PAPERS_DIR / f"{arxiv_id}.txt"
    if txt_path.exists():
        print(f"[ingesting] {arxiv_id}")
        
        subprocess.run(
            [sys.executable, "-m", "kb_mcp.kb.cli", "ingest", str(txt_path),
             "--source-id", "arxiv", "--no-summary", "--batch"],
            capture_output=True
        )

print("\nDone! Checking database...")
result = subprocess.run([sys.executable, "-m", "kb_mcp.kb.cli", "stats"], capture_output=True, text=True)
print(result.stdout)

[ingesting] 1807.06209
[ingesting] 2007.08991
[ingesting] 1502.01589

Done! Checking database...
Knowledge Base Statistics
Total documents: 3
Total sources: 1

Documents by source:
  arxiv: 3



### Set up HEP Multiagent

Set up the LLM

In [6]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="claudesonnet4",
    base_url="https://apps-dev.inside.anl.gov/argoapi/v1",
    api_key=os.environ["ARGO_USER"]
)
print("Using Argo API")

/Users/celsloaner/Desktop/hep-multiagent-v2/hep-multiagent/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using Argo API


Initialize hep-multiagent with kb-mcp. The agent gets these tools:
- `kb_search` - search papers by keyword
- `kb_get` - get full text of a paper

In [7]:
from hep_multiagent import Agent

agent = await Agent(
    llm=llm,
    mcp_servers=[
    {
        "url": "https://github.com/HEP-KE/kb-mcp.git",
        "name": "kb-server-stdio",
        "env": {"SQLITE_DB_PATH": str(DB_PATH)},
    },
    {
        "url": "https://github.com/HEP-KE/mcp-ke.git",
    },
    ],
    approval=False,
)

for tool in agent.tools:
    print(f"  - {tool.name}")

  - kb_search
  - kb_get
  - kb_get_image
  - list_agent_files
  - load_array
  - load_dict
  - save_array
  - save_dict
  - compute_all_models
  - compute_power_spectrum
  - compute_suppression_ratios
  - get_lcdm_params
  - get_nu_mass_params
  - get_wcdm_params
  - create_theory_k_grid
  - load_observational_data
  - plot_power_spectra
  - plot_suppression_ratios
  - arxiv_agent
  - download_arxiv_paper
  - download_full_arxiv_paper
  - list_files
  - read_text_file
  - search_arxiv
  - power_spectrum_agent


## Query

In [8]:
result = await agent.run(
    query=" Mock observational data from eBOSS DR14 Lyman-alpha forest then compute theoretical cosmology models (ΛCDM, neutrinos, wCDM), and create comparison visualizations",
    output_dir="./output_kb_hep_mcp"
)

print(result.get("final_report", "No report")[:2000])

CancelledError: 